Use the attached cancer dataset (cancer_reg.csv Download cancer_reg.csv) to develop a machine learning model to predict target_deathrate using train/test split methodology. Data details are at Data Dictionary.docx Download Data Dictionary.docx.

Set up model monitoring with https://www.evidentlyai.com/Links to an external site..

Evaluate your model accuracy with the test dataset.

Next make the following changes to your test dataset one at a time

A. Change the medianincome by decreasing by 40,000 (eg.. 43,823 becomes 3,823)

B. Change the povertypercent by increasing it by 20 points (eg 11.9 changes to 31.9).

C. Change the avghouseholdsize by increasing it by 2 (eg. 2.31 becomes 4.31).

Run 1 instance with A, another with A & B, and a third with A & B & C.

For each instance, run the model verify the accuracy, and use model monitoring to detect the changes in the input and the model output.

Submit code for the above, with (Optional) Evidently dashboard screenhsot on the platform https://www.evidentlyai.com/Links to an external site. with your token


In [ ]:
## Pip install Evidently

!pip install evidently

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.8/237.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.0 MB/s eta 0:00:00


In [ ]:
## Check Version

import evidently
print(evidently.__version__)

0.7.17


In [ ]:
## Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset, RegressionPreset

from evidently.ui.workspace import CloudWorkspace

In [ ]:
## Create Workspace

ws = CloudWorkspace(token="dG9rbgF4ZdZyRdJJp4fPDfRBoazAYFXc8E9LCelDi5UQ+gMLAwBQjiW+dUQsFnKqrHpKRt2pTuhQCh+ZoJwm/mM94wEyoCx0nTkk5aifvfbv7apR9sIgDErvQSfSsIxO7xQqv3myW2lHtC+iGaWaTzKKyBxmv5Np7rPO", url="https://app.evidently.cloud")

In [ ]:
## Create Project

project = ws.create_project("MLOps Assignment 4", org_id="019a7fdd-3d0e-7937-b171-052863cbaa88")
project.description = "machine learning model to predict target_deathrate using train/test split methodology"
project.save()

In [ ]:
## Load Data

data = pd.read_csv("cancer_reg.csv", encoding="latin1")
data

,avgAnnCount,avgDeathsPerYear,TARGET_deathRate,incidenceRate,medIncome,popEst2015,povertyPercent,studyPerCap,binnedInc,MedianAge,...,PctPrivateCoverageAlone,PctEmpPrivCoverage,PctPublicCoverage,PctPublicCoverageAlone,PctWhite,PctBlack,PctAsian,PctOtherRace,PctMarriedHouseholds,BirthRate
0,1397.000000,469,164.9,489.800000,61898,260131,11.2,499.748204,"(61494.5, 125635]",39.3,...,NaN,41.6,32.9,14.0,81.780529,2.594728,4.821857,1.843479,52.856076,6.118831
1,173.000000,70,161.3,411.600000,48127,43269,18.6,23.111234,"(48021.6, 51046.4]",33.0,...,53.8,43.6,31.1,15.3,89.228509,0.969102,2.246233,3.741352,45.372500,4.333096
2,102.000000,50,174.7,349.700000,49348,21026,14.6,47.560164,"(48021.6, 51046.4]",45.0,...,43.5,34.9,42.1,21.1,90.922190,0.739673,0.465898,2.747358,54.444868,3.729488
3,427.000000,202,194.8,430.400000,44243,75882,17.1,342.637253,"(42724.4, 45201]",42.8,...,40.3,35.0,45.3,25.0,91.744686,0.782626,1.161359,1.362643,51.021514,4.603841
4,57.000000,26,144.4,350.100000,49955,10321,12.5,0.000000,"(48021.6, 51046.4]",48.3,...,43.9,35.1,44.0,22.7,94.104024,0.270192,0.665830,0.492135,54.027460,6.796657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3042,1962.667684,15,149.6,453.549422,46961,6343,12.4,0.000000,"(45201, 48021.6]",44.2,...,54.9,44.6,31.7,13.2,90.280811,3.837754,0.327613,1.700468,51.063830,7.773512
3043,1962.667684,43,150.1,453.549422,48609,37118,18.8,377.175494,"(48021.6, 51046.4]",30.4,...,53.3,48.6,28.8,17.7,75.706245,2.326771,4.044920,14.130288,52.007937,8.186470
3044,1962.667684,46,153.9,453.549422,51144,34536,15.0,1968.959926,"(51046.4, 54545.6]",30.9,...,52.6,47.8,26.6,16.8,87.961629,2.313188,1.316472,5.680705,55.153949,7.809192
3045,1962.667684,52,175.0,453.549422,50745,25609,13.3,0.000000,"(48021.6, 51046.4]",39.0,...,56.3,49.6,29.5,14.0,92.905681,1.176562,0.244632,2.131790,58.484232,7.582938


In [ ]:
## Clean Data

def midpoint_from_bin(bin_str):
  bin_str = bin_str.strip('()[]')
  low, high = bin_str.split(',')
  return (float(low) + float(high)) / 2

if 'binnedInc' in data.columns:
  data['binnedInc'] = data['binnedInc'].apply(midpoint_from_bin)

# if 'Geography' in data.columns:
#   data = pd.get_dummies(data, columns=['Geography'], drop_first=True)

In [ ]:
## Check Columns

data.columns

Index(['avgAnnCount', 'avgDeathsPerYear', 'TARGET_deathRate', 'incidenceRate',
       'medIncome', 'popEst2015', 'povertyPercent', 'studyPerCap', 'binnedInc',
       'MedianAge', 'MedianAgeMale', 'MedianAgeFemale', 'Geography',
       'AvgHouseholdSize', 'PercentMarried', 'PctNoHS18_24', 'PctHS18_24',
       'PctSomeCol18_24', 'PctBachDeg18_24', 'PctHS25_Over',
       'PctBachDeg25_Over', 'PctEmployed16_Over', 'PctUnemployed16_Over',
       'PctPrivateCoverage', 'PctPrivateCoverageAlone', 'PctEmpPrivCoverage',
       'PctPublicCoverage', 'PctPublicCoverageAlone', 'PctWhite', 'PctBlack',
       'PctAsian', 'PctOtherRace', 'PctMarriedHouseholds', 'BirthRate'],
      dtype='object')

In [ ]:
target = "TARGET_deathRate"

# Remove rows missing target
data = data.dropna(subset=[target])

# Select numeric features only
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove(target)

# Fill missing numeric values
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

In [ ]:
# ## Schema Def
# schema = DataDefinition(
#     numerical_columns=['avgAnnCount', 'avgDeathsPerYear', 'TARGET_deathRate', 'incidenceRate',
#        'medIncome', 'popEst2015', 'povertyPercent', 'studyPerCap', 'binnedInc',
#        'MedianAge', 'MedianAgeMale', 'MedianAgeFemale',
#        'AvgHouseholdSize', 'PercentMarried', 'PctNoHS18_24', 'PctHS18_24',
#        'PctSomeCol18_24', 'PctBachDeg18_24', 'PctHS25_Over',
#        'PctBachDeg25_Over', 'PctEmployed16_Over', 'PctUnemployed16_Over',
#        'PctPrivateCoverage', 'PctPrivateCoverageAlone', 'PctEmpPrivCoverage',
#        'PctPublicCoverage', 'PctPublicCoverageAlone', 'PctWhite', 'PctBlack',
#        'PctAsian', 'PctOtherRace', 'PctMarriedHouseholds', 'BirthRate'],
#     categorical_columns=["Geography"],
#     )

In [ ]:
## Get Categorical Variables & Fill Nulls with mean

# if 'Geography' in data.columns:
#   data = pd.get_dummies(data, columns=['Geography'], drop_first=True)

# data.fillna(data.mean())

,avgAnnCount,avgDeathsPerYear,TARGET_deathRate,incidenceRate,medIncome,popEst2015,povertyPercent,studyPerCap,binnedInc,MedianAge,...,"Geography_York County, Pennsylvania","Geography_York County, South Carolina","Geography_York County, Virginia","Geography_Young County, Texas","Geography_Yuba County, California","Geography_Yukon-Koyukuk Census Area, Alaska","Geography_Yuma County, Arizona","Geography_Yuma County, Colorado","Geography_Zapata County, Texas","Geography_Zavala County, Texas"
0,1397.000000,469,164.9,489.800000,61898,260131,11.2,499.748204,93564.75,39.3,...,False,False,False,False,False,False,False,False,False,False
1,173.000000,70,161.3,411.600000,48127,43269,18.6,23.111234,49534.00,33.0,...,False,False,False,False,False,False,False,False,False,False
2,102.000000,50,174.7,349.700000,49348,21026,14.6,47.560164,49534.00,45.0,...,False,False,False,False,False,False,False,False,False,False
3,427.000000,202,194.8,430.400000,44243,75882,17.1,342.637253,43962.70,42.8,...,False,False,False,False,False,False,False,False,False,False
4,57.000000,26,144.4,350.100000,49955,10321,12.5,0.000000,49534.00,48.3,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3042,1962.667684,15,149.6,453.549422,46961,6343,12.4,0.000000,46611.30,44.2,...,False,False,False,False,False,False,False,False,False,False
3043,1962.667684,43,150.1,453.549422,48609,37118,18.8,377.175494,49534.00,30.4,...,False,False,False,False,False,False,False,False,False,False
3044,1962.667684,46,153.9,453.549422,51144,34536,15.0,1968.959926,52796.00,30.9,...,False,False,False,False,False,False,False,False,False,False
3045,1962.667684,52,175.0,453.549422,50745,25609,13.3,0.000000,49534.00,39.0,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
categorical_cols = ["Geography"]
numeric_cols = data.select_dtypes(include=["number"]).columns.tolist()
numeric_cols.remove(target)

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=False)

In [ ]:
X = data_encoded.drop(columns=[target])
y = data_encoded[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

X_train.shape, X_test.shape

((2285, 3079), (762, 3079))

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

def metrics(true, pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(true, pred)),
        "MAE": mean_absolute_error(true, pred),
        "R2": r2_score(true, pred)
    }

baseline_metrics = metrics(y_test, y_pred)
baseline_metrics

{'RMSE': np.float64(18.919097701847043),
 'MAE': 13.945070209973752,
 'R2': 0.5548152478627248}

In [ ]:
test_a = X_test.copy()
test_a["medIncome"] = test_a["medIncome"] - 40000

test_ab = test_a.copy()
test_ab["povertyPercent"] = test_ab["povertyPercent"] + 20

test_abc = test_ab.copy()
test_abc["AvgHouseholdSize"] = test_abc["AvgHouseholdSize"] + 2

In [ ]:
# Predictions

results = {}

for name, Xt in [
    ("Baseline", X_test),
    ("A", test_a),
    ("A+B", test_ab),
    ("A+B+C", test_abc)
]:
    preds = model.predict(Xt)
    results[name] = metrics(y_test, preds)

results


{'Baseline': {'RMSE': np.float64(18.919097701847043),
  'MAE': 13.945070209973752,
  'R2': 0.5548152478627248},
 'A': {'RMSE': np.float64(20.1392864182022),
  'MAE': 15.30943044619423,
  'R2': 0.49553899830198056},
 'A+B': {'RMSE': np.float64(21.242190279722127),
  'MAE': 16.40266010498687,
  'R2': 0.4387736793027268},
 'A+B+C': {'RMSE': np.float64(20.699052672272888),
  'MAE': 15.961409448818888,
  'R2': 0.4671065521646991}}

In [ ]:
reference_data = pd.concat([X_train, y_train.rename(target)], axis=1)
reference_data.head()

,avgAnnCount,avgDeathsPerYear,incidenceRate,medIncome,popEst2015,povertyPercent,studyPerCap,binnedInc,MedianAge,MedianAgeMale,...,"Geography_York County, South Carolina","Geography_York County, Virginia","Geography_Young County, Texas","Geography_Yuba County, California","Geography_Yukon-Koyukuk Census Area, Alaska","Geography_Yuma County, Arizona","Geography_Yuma County, Colorado","Geography_Zapata County, Texas","Geography_Zavala County, Texas",TARGET_deathRate
1316,1962.667684,6,453.549422,49852,2825,9.7,0.000000,49534.00,42.2,40.6,...,False,False,False,False,False,False,False,False,False,129.1
1229,178.000000,83,478.600000,34526,34167,25.9,0.000000,35815.95,36.5,35.2,...,False,False,False,False,False,False,False,False,False,224.4
2676,48.000000,23,372.400000,31829,10886,29.1,0.000000,28429.05,37.1,36.0,...,False,False,False,False,False,False,False,False,False,174.8
1117,202.000000,92,376.600000,46371,43664,15.2,0.000000,46611.30,40.6,39.8,...,False,False,False,False,False,False,False,False,False,170.7
2457,1962.667684,111,453.549422,88500,98741,4.8,638.032833,93564.75,36.9,36.1,...,False,False,False,False,False,False,False,False,False,148.8


In [ ]:
reference_data = X_train.copy()
reference_data['TARGET_deathRate'] = y_train.values

current_data_a = test_a.copy()
current_data_a['TARGET_deathRate'] = y_test.values

report_a = Report(metrics=[DataDriftPreset()])
report_a.run(reference_data=reference_data, current_data=current_data_a)
report_a.save_html('drift_report_scenario_A.html')


In [38]:
reference_data = X_train.copy()
reference_data['TARGET_deathRate'] = y_train.values

current_data_ab = test_ab.copy()
current_data_ab['TARGET_deathRate'] = y_test.values

report_ab = Report(metrics=[DataDriftPreset()])
report_ab.run(reference_data=reference_data, current_data=current_data_ab)
report_ab.save_html('drift_report_scenario_AB.html')


Available submodules:
evidently.core.registries.column_conditions
evidently.generators.column
evidently.legacy.metrics.data_drift.column_drift_metric
evidently.legacy.metrics.data_drift.column_interaction_plot
evidently.legacy.metrics.data_drift.column_value_plot
evidently.legacy.metrics.data_integrity.column_missing_values_metric
evidently.legacy.metrics.data_integrity.column_regexp_metric
evidently.legacy.metrics.data_integrity.column_summary_metric
evidently.legacy.metrics.data_quality.column_category_metric
evidently.legacy.metrics.data_quality.column_correlations_metric
evidently.legacy.metrics.data_quality.column_distribution_metric
evidently.legacy.metrics.data_quality.column_quantile_metric
evidently.legacy.metrics.data_quality.column_value_list_metric
evidently.legacy.metrics.data_quality.column_value_range_metric
evidently.legacy.pipeline.column_mapping
evidently.metrics.column_statistics


In [ ]:
reference_data = X_train.copy()
reference_data['TARGET_deathRate'] = y_train.values

current_data_abc = test_abc.copy()
current_data_abc['TARGET_deathRate'] = y_test.values

report_abc = Report(metrics=[DataDriftPreset()])
report_abc.run(reference_data=reference_data, current_data=current_data_abc)
report_abc.save_html('drift_report_scenario_ABC.html')